In [9]:
"""
=============================================================================
Multi-Agent Chaos Engineering - 5 Agent Architecture Example
=============================================================================
Reference: https://github.com/aws-samples/sample-strands-chaos-engineering-agents

Pipeline:
  Hypothesis Generator → Prioritization → Experiment Design → Execution → Learning

=============================================================================
"""
from dotenv import load_dotenv
load_dotenv()

import os
import json
from datetime import datetime

def print_cell_header(cell_name):
    now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    print(f"⏱️  [{now}] {cell_name}")
    print("-" * 60)


print_cell_header("Cell 0: 환경 설정 확인")
print("Region:", "✅ 설정됨" if os.getenv("AWS_DEFAULT_REGION") else "❌ 미설정")
print("Bearer Token:", "✅ 설정됨" if os.getenv("AWS_BEARER_TOKEN_BEDROCK") else "❌ 미설정")
print("\n✅ 환경 설정 확인 완료")

⏱️  [2026-06-06 15:45:30] Cell 0: 환경 설정 확인
------------------------------------------------------------
Region: ✅ 설정됨
Bearer Token: ✅ 설정됨

✅ 환경 설정 확인 완료


In [ ]:
"""
=============================================================================
Cell 1: 공통 도구 정의 (실제 AWS 연동)
=============================================================================
boto3를 사용하여 실제 EKS/FIS API를 호출합니다.
"""
print_cell_header("Cell 1: AWS 도구 초기화")
import subprocess
import boto3
from strands import tool
from strands.models.openai import OpenAIModel

# Bedrock 모델 (API Key → Mantle OpenAI-compatible endpoint)
bedrock_model = OpenAIModel(
    client_args={
        "base_url": "https://bedrock-mantle.us-east-1.api.aws/v1",
        "api_key": os.getenv("AWS_BEARER_TOKEN_BEDROCK"),
    },
    model_id="qwen.qwen3-coder-30b-a3b-instruct"
)

# AWS 세션 (EKS/FIS용)
aws_session = boto3.Session(profile_name=os.getenv("AWS_PROFILE_EKS", os.getenv("AWS_PROFILE_EKS")))
eks_region = os.getenv("EKS_REGION", "ap-northeast-2")
cluster_name = os.getenv("EKS_CLUSTER_NAME")
namespace = os.getenv("EKS_NAMESPACE", "retail-store")
fis_role_arn = os.getenv("FIS_ROLE_ARN", "")

fis_client = aws_session.client("fis", region_name=eks_region)
eks_client = aws_session.client("eks", region_name=eks_region)


@tool
def discover_eks_resources(target_namespace: str = "retail-store") -> str:
    """EKS 클러스터의 리소스(Pod, Service, Deployment)를 탐색합니다.
    Args:
        target_namespace: 탐색할 Kubernetes 네임스페이스
    """
    try:
        result = subprocess.run(
            ["kubectl", "get", "all", "-n", target_namespace, "-o", "json"],
            capture_output=True, text=True, timeout=30
        )
        if result.returncode != 0:
            return f"Error: {result.stderr}"

        import json as _json
        data = _json.loads(result.stdout)
        
        summary = {"namespace": target_namespace, "deployments": [], "statefulsets": [], "pods": [], "services": []}
        
        for item in data.get("items", []):
            kind = item.get("kind", "")
            name = item.get("metadata", {}).get("name", "")
            
            if kind == "Deployment":
                spec = item.get("spec", {})
                replicas = spec.get("replicas", 0)
                containers = spec.get("template", {}).get("spec", {}).get("containers", [])
                images = [c.get("image", "") for c in containers]
                summary["deployments"].append({"name": name, "replicas": replicas, "images": images})
            elif kind == "StatefulSet":
                spec = item.get("spec", {})
                replicas = spec.get("replicas", 0)
                volume_claims = spec.get("volumeClaimTemplates", [])
                has_pvc = len(volume_claims) > 0
                summary["statefulsets"].append({"name": name, "replicas": replicas, "persistent_storage": has_pvc})
            elif kind == "Pod":
                status = item.get("status", {}).get("phase", "Unknown")
                summary["pods"].append({"name": name, "status": status})
            elif kind == "Service":
                svc_type = item.get("spec", {}).get("type", "ClusterIP")
                summary["services"].append({"name": name, "type": svc_type})

        return json.dumps(summary, indent=2, ensure_ascii=False)
    except Exception as e:
        return f"Error discovering resources: {str(e)}"


@tool
def get_fis_actions(service_filter: str = "eks") -> str:
    """사용 가능한 AWS FIS 액션 목록을 가져옵니다.
    Args:
        service_filter: 필터링할 AWS 서비스 (eks, ec2, rds 등)
    """
    try:
        response = fis_client.list_actions()
        actions = []
        for action in response.get("actions", []):
            if service_filter in action["id"]:
                actions.append({
                    "id": action["id"],
                    "description": action.get("description", "")
                })
        return json.dumps(actions, indent=2, ensure_ascii=False)
    except Exception as e:
        return f"Error listing FIS actions: {str(e)}"


@tool
def create_fis_experiment(template_name: str, action_id: str, target_namespace: str, target_selector: str, duration_minutes: int = 5) -> str:
    """AWS FIS 실험 템플릿을 생성합니다.
    Args:
        template_name: 실험 템플릿 이름
        action_id: FIS 액션 ID (예: aws:eks:pod-delete)
        target_namespace: 대상 K8s 네임스페이스
        target_selector: 대상 Pod 라벨 셀렉터 (예: app.kubernetes.io/name=orders)
        duration_minutes: 실험 지속 시간(분)
    """
    try:
        cluster_arn = f"arn:aws:eks:{eks_region}:{aws_session.client('sts').get_caller_identity()['Account']}:cluster/{cluster_name}"

        action_params = {
            "kubernetesServiceAccount": "default",
        }
        if "stress" in action_id:
            action_params["duration"] = f"PT{duration_minutes}M"

        response = fis_client.create_experiment_template(
            description=f"Chaos experiment: {template_name}",
            roleArn=fis_role_arn,
            actions={
                "action-1": {
                    "actionId": action_id,
                    "parameters": action_params,
                    "targets": {
                        "Pods": "target-pods"
                    }
                }
            },
            targets={
                "target-pods": {
                    "resourceType": "aws:eks:pod",
                    "parameters": {
                        "clusterIdentifier": cluster_arn,
                        "namespace": target_namespace,
                        "selectorType": "labelSelector",
                        "selectorValue": target_selector
                    },
                    "selectionMode": "ALL"
                }
            },
            stopConditions=[
                {"source": "none"}
            ],
            tags={
                "Name": template_name,
                "Project": "chaos-engineering",
                "ManagedBy": "strands-agent"
            }
        )
        template_id = response["experimentTemplate"]["id"]
        return json.dumps({
            "status": "CREATED",
            "template_id": template_id,
            "template_name": template_name,
            "action": action_id,
            "target": f"{target_namespace}/{target_selector}"
        }, ensure_ascii=False)
    except Exception as e:
        return f"Error creating FIS template: {str(e)}"


@tool
def run_fis_experiment(experiment_template_id: str) -> str:
    """AWS FIS 실험을 실행하고 결과를 반환합니다.
    Args:
        experiment_template_id: 실행할 FIS 실험 템플릿 ID
    """
    try:
        response = fis_client.start_experiment(
            experimentTemplateId=experiment_template_id,
            tags={"RunBy": "strands-chaos-agent"}
        )
        experiment_id = response["experiment"]["id"]
        status = response["experiment"]["state"]["status"]
        
        return json.dumps({
            "experiment_id": experiment_id,
            "template_id": experiment_template_id,
            "status": status,
            "message": f"실험 시작됨. FIS 콘솔에서 확인: https://{eks_region}.console.aws.amazon.com/fis/home?region={eks_region}#/experiments/{experiment_id}"
        }, ensure_ascii=False)
    except Exception as e:
        return f"Error running experiment: {str(e)}"


@tool
def check_fis_experiment_status(experiment_id: str) -> str:
    """실행 중인 FIS 실험의 상태를 확인합니다.
    Args:
        experiment_id: 확인할 실험 ID
    """
    try:
        response = fis_client.get_experiment(id=experiment_id)
        exp = response["experiment"]
        return json.dumps({
            "experiment_id": exp["id"],
            "status": exp["state"]["status"],
            "reason": exp["state"].get("reason", ""),
            "start_time": str(exp.get("startTime", "")),
            "end_time": str(exp.get("endTime", "")),
        }, ensure_ascii=False)
    except Exception as e:
        return f"Error checking experiment: {str(e)}"


@tool
def save_to_database(table: str, data: str) -> str:
    """실험 데이터를 로컬 JSON 파일에 저장합니다.
    Args:
        table: 데이터 종류 (hypothesis, experiment, result)
        data: 저장할 데이터
    """
    import pathlib
    output_dir = pathlib.Path("output")
    output_dir.mkdir(exist_ok=True)
    
    filepath = output_dir / f"{table}.json"
    with open(filepath, "a", encoding="utf-8") as f:
        f.write(data + "\n---\n")
    
    return f"✅ [{table}] 데이터 저장 완료 → {filepath}"


print(f"✅ 실제 AWS 도구 6개 정의 완료")
print(f"   클러스터: {cluster_name} ({eks_region})")
print(f"   네임스페이스: {namespace}")
print(f"   FIS Role: {'✅ 설정됨' if fis_role_arn else '❌ 미설정'}")

⏱️  [2026-06-06 15:45:32] Cell 1: AWS 도구 초기화
------------------------------------------------------------
✅ 실제 AWS 도구 6개 정의 완료
   클러스터: *** (ap-northeast-2)
   네임스페이스: retail-store
   FIS Role: ✅ 설정됨


In [11]:
"""
=============================================================================
Cell 2: Agent 1 - Hypothesis Generator (가설 생성 에이전트)
=============================================================================
"""
print_cell_header("Cell 2: Agent 1 - Hypothesis Generator")
from strands import Agent
from IPython.display import display, HTML
import re

HYPOTHESIS_PROMPT = (
    "당신은 카오스 엔지니어링 가설 생성 전문가입니다.\n"
    "역할: AWS EKS 워크로드를 분석하여 장애 가설을 생성합니다.\n"
    "반드시 JSON 배열만 출력하세요:\n"
    '[{"id":"H-001","title":"제목","service":"서비스","failure_domain":"compute|data|network|dependency|resource",'
    '"hypothesis":"만약 X하면 Y","description":"배경설명 2-3문장","steady_state":"정상 상태","expected_result":"예상 결과","impact_score":1,"likelihood_score":1,'
    '"suggested_fis_action":"액션ID","label_selector":"app.kubernetes.io/name=서비스"}]\n'
    "규칙:\n"
    "- 5개 생성, 각 도메인(compute,data,network,dependency,resource) 1개씩\n"
    "- description: 가설의 배경과 이 테스트가 중요한 이유를 2-3문장으로 설명\n"
    "- steady_state: 장애가 없을 때의 정상 동작 정의\n"
    "- expected_result: 장애 주입 시 예상되는 구체적 결과\n"
    "- label_selector는 실제 발견된 서비스의 라벨을 사용\n"
    "- 최종 응답은 JSON 배열만 출력"
)

hypothesis_agent = Agent(
    model=bedrock_model,
    system_prompt=HYPOTHESIS_PROMPT,
    tools=[discover_eks_resources, get_fis_actions, save_to_database],
    callback_handler=None
)

print("🔬 [Agent 1] Hypothesis Generator 실행 중...\n")
hypothesis_result = hypothesis_agent(
    "retail-store 네임스페이스 워크로드를 discover_eks_resources로 탐색하고, "
    "카오스 엔지니어링 가설 5개를 JSON 배열로 생성한 뒤 "
    "save_to_database hypothesis 테이블에 저장하세요."
)

# 결과 파싱
h_text = hypothesis_result.message["content"][0]["text"] if hypothesis_result.message["content"] else "[]"
json_match = re.search(r'\[.*\]', h_text, re.DOTALL)
hypotheses_data = json.loads(json_match.group()) if json_match else []
token_count = hypothesis_result.metrics.accumulated_usage.get("totalTokens", 0)

# 시각화
DOMAIN_CFG = {
    "compute": ("\U0001f5a5\ufe0f", "#ff6b6b", "컴퓨트"),
    "data": ("\U0001f4be", "#feca57", "데이터"),
    "network": ("\U0001f310", "#48dbfb", "네트워크"),
    "dependency": ("\U0001f517", "#ff9ff3", "종속성"),
    "resource": ("\u2699\ufe0f", "#54a0ff", "리소스"),
}

tags_html = ""
for icon, color, label in DOMAIN_CFG.values():
    tags_html += (
        f'<span style="background:{color}22; color:{color}; '
        f'padding:4px 10px; border-radius:6px; font-size:11px;">'
        f'{icon} {label}</span> '
    )

cards_html = ""
for h in hypotheses_data:
    domain = h.get("failure_domain", "compute")
    icon, color, label = DOMAIN_CFG.get(domain, DOMAIN_CFG["compute"])
    impact = h.get("impact_score", 5)
    likelihood = h.get("likelihood_score", 5)
    risk = impact * 0.6 + likelihood * 0.4
    rc = "#ff6b6b" if risk >= 7 else "#feca57" if risk >= 5 else "#48dbfb"
    cards_html += (
        f'<div style="background:rgba(255,255,255,0.03); border:1px solid {color}33; '
        f'border-left:4px solid {color}; border-radius:8px; padding:14px 18px; margin:8px 0;">'
        f'<div style="display:flex; justify-content:space-between; align-items:center;">'
        f'<div><span style="background:{color}22; color:{color}; padding:2px 8px; '
        f'border-radius:4px; font-size:11px; font-weight:600;">{icon} {label}</span>'
        f'<span style="color:#666; font-size:11px; margin-left:8px;">{h.get("id","")}</span></div>'
        f'<div style="background:{rc}22; color:{rc}; padding:3px 10px; '
        f'border-radius:12px; font-size:12px; font-weight:700;">Risk {risk:.1f}</div></div>'
        f'<div style="font-size:14px; font-weight:600; margin:8px 0 4px; color:#e0e0e0;">'
        f'{h.get("title","")}</div>'
        f'<div style="font-size:12px; color:#999; margin-bottom:10px;">'
        f'{h.get("hypothesis","")}</div>'
        f'<div style="font-size:11px; color:#888; margin:6px 0 10px; padding:8px 10px; background:rgba(255,255,255,0.02); border-radius:4px; line-height:1.6;">'
        f'<div style="margin-bottom:3px;"><b style="color:#aaa;">📋 배경:</b> {h.get("description", "-")}</div>'
        f'<div style="margin-bottom:3px;"><b style="color:#aaa;">✅ 정상 상태:</b> {h.get("steady_state", "-")}</div>'
        f'<div><b style="color:#aaa;">⚠️ 예상 결과:</b> {h.get("expected_result", "-")}</div>'
        f'</div>'
        f'<div style="display:flex; gap:16px; font-size:11px;">'
        f'<div style="flex:1;"><div style="color:#888;">Impact</div>'
        f'<div style="background:rgba(255,255,255,0.1); border-radius:4px; height:6px;">'
        f'<div style="width:{impact*10}%; height:100%; background:#ff6b6b; border-radius:4px;">'
        f'</div></div><div style="color:#ff6b6b; font-weight:600;">{impact}/10</div></div>'
        f'<div style="flex:1;"><div style="color:#888;">Likelihood</div>'
        f'<div style="background:rgba(255,255,255,0.1); border-radius:4px; height:6px;">'
        f'<div style="width:{likelihood*10}%; height:100%; background:#feca57; border-radius:4px;">'
        f'</div></div><div style="color:#feca57; font-weight:600;">{likelihood}/10</div></div>'
        f'<div style="flex:1;"><div style="color:#888;">Target</div>'
        f'<div style="color:#48dbfb; font-family:monospace; font-size:11px;">'
        f'{h.get("service","")}</div></div>'
        f'<div style="flex:1;"><div style="color:#888;">FIS Action</div>'
        f'<div style="color:#54a0ff; font-family:monospace; font-size:10px;">'
        f'{h.get("suggested_fis_action","")}</div></div>'
        f'</div></div>'
    )

full_html = (
    '<div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif; '
    'background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); '
    'color:#fff; padding:24px; border-radius:14px; margin:10px 0;">'
    '<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">'
    '<h3 style="margin:0; font-size:18px;">🔬 Agent 1: Hypothesis Generator</h3>'
    f'<span style="background:rgba(72,219,251,0.15); color:#48dbfb; '
    f'padding:4px 12px; border-radius:12px; font-size:12px;">✅ {len(hypotheses_data)}개 생성</span></div>'
    f'<div style="display:flex; gap:8px; margin-bottom:16px; flex-wrap:wrap;">{tags_html}</div>'
    f'{cards_html}'
    f'<div style="margin-top:14px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); '
    f'font-size:11px; color:#666; text-align:right;">Tokens: {token_count:,} | Model: qwen3-coder-30b</div>'
    '</div>'
)

display(HTML(full_html))


⏱️  [2026-06-06 15:45:34] Cell 2: Agent 1 - Hypothesis Generator
------------------------------------------------------------
🔬 [Agent 1] Hypothesis Generator 실행 중...



In [12]:
"""
=============================================================================
Cell 3: Agent 2 - Prioritization Agent (우선순위 결정 에이전트)
=============================================================================
생성된 가설들의 영향도/가능성 기반 우선순위를 결정합니다.
"""

print_cell_header("Cell 3: Agent 2 - Prioritization")
prioritization_agent = Agent(
    model=bedrock_model,
    system_prompt="""당신은 카오스 엔지니어링 가설 우선순위 결정 전문가입니다.

역할:
- 생성된 가설들을 분석하여 실험 우선순위를 결정합니다.
- Safety-first 원칙에 따라 폭발 반경이 작은 실험부터 시작합니다.

우선순위 결정 기준 (가중치):
1. 영향도 (Impact): 장애 시 비즈니스 영향 (40%)
2. 가능성 (Likelihood): 실제 발생 가능성 (25%)
3. 안전성 (Safety): 실험의 안전한 실행 가능성 - 안전할수록 높은 순위 (20%)
4. 학습 가치 (Learning Value): 실험에서 얻을 수 있는 인사이트 (15%)

출력 형식:
{
    "prioritized_hypotheses": [
        {
            "id": "H-XXX",
            "priority": 1,
            "title": "...",
            "scores": {"impact": 8, "likelihood": 7, "safety": 9, "learning_value": 8},
            "total_score": 8.1,
            "rationale": "우선순위 결정 이유"
        }
    ]
}
""",
    tools=[save_to_database],
    callback_handler=None
)

# 이전 에이전트 결과를 입력으로 사용
hypothesis_output = hypothesis_result.message["content"][0]["text"] if hypothesis_result.message else ""

print("📊 [Agent 2] Prioritization Agent 실행 중...\n")
priority_result = prioritization_agent(
    f"""다음 가설들을 우선순위에 따라 정렬해주세요.
Safety-first 원칙에 따라, 폭발 반경이 작고 롤백이 명확한 실험이 높은 우선순위를 가집니다.

가설 목록:
{hypothesis_output}

모든 가설에 대해 점수를 매기고 우선순위를 결정한 후,
결과를 save_to_database 도구로 hypothesis 테이블에 업데이트해주세요."""
)

# --- 우선순위 시각화 ---
p_text = priority_result.message["content"][0]["text"] if priority_result.message["content"] else "[]"
p_match = re.search(r'\[.*\]', p_text, re.DOTALL)
priority_data = json.loads(p_match.group()) if p_match else []
if not priority_data:
    priority_data = sorted(hypotheses_data, key=lambda x: (x.get("impact_score",0)*0.4 + x.get("likelihood_score",0)*0.25 + 8*0.2 + 7*0.15), reverse=True)

sorted_priorities = sorted(priority_data, key=lambda x: x.get("total_score", x.get("impact_score",0)*0.6+x.get("likelihood_score",0)*0.4), reverse=True)

rank_rows = ""

for i, p in enumerate(sorted_priorities[:5]):
    scores = p.get("scores", {})
    total = p.get("total_score", p.get("impact_score",0)*0.6 + p.get("likelihood_score",0)*0.4)
    bar_pct = total * 10
    bar_color = "#ff6b6b" if total >= 8 else "#feca57" if total >= 6.5 else "#48dbfb"
    rank_label = ["🥇", "🥈", "🥉"][i] if i < 3 else str(i+1)
    rank_rows += f"""
    <tr style="border-bottom:1px solid rgba(255,255,255,0.05);">
        <td style="padding:12px 8px; font-size:24px; text-align:center;">{rank_label}</td>
        <td style="padding:12px 8px;">
            <div style="font-weight:600; font-size:13px;">{p.get('title', p.get('id',''))}</div>
            <div style="font-size:11px; color:#888; margin-top:2px;">{p.get('rationale', p.get('hypothesis',''))[:80]}</div>
        </td>
        <td style="padding:12px 8px; text-align:center;">
            <div style="font-size:18px; font-weight:700; color:{bar_color};">{total:.1f}</div>
        </td>
        <td style="padding:12px 8px; width:25%;">
            <div style="background:rgba(255,255,255,0.08); border-radius:6px; height:10px; overflow:hidden;">
                <div style="width:{bar_pct}%; height:100%; background:linear-gradient(90deg, {bar_color}, {bar_color}88); border-radius:6px;"></div>
            </div>
        </td>
    </tr>"""

priority_html = f"""
<div style="font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',sans-serif; background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); color:#fff; padding:24px; border-radius:14px; margin:10px 0;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">
        <h3 style="margin:0; font-size:18px;">📊 Agent 2: Prioritization</h3>
        <span style="background:rgba(254,202,87,0.15); color:#feca57; padding:4px 12px; border-radius:12px; font-size:12px;">Safety-First Ranking</span>
    </div>
    <table style="width:100%; border-collapse:collapse;">
        <tr style="border-bottom:1px solid rgba(255,255,255,0.1);">
            <th style="padding:8px; font-size:11px; color:#888; text-align:left;">Rank</th>
            <th style="padding:8px; font-size:11px; color:#888; text-align:left;">Hypothesis</th>
            <th style="padding:8px; font-size:11px; color:#888; text-align:center;">Score</th>
            <th style="padding:8px; font-size:11px; color:#888; text-align:left;">Priority</th>
        </tr>
        {rank_rows}
    </table>
    <div style="margin-top:14px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); font-size:11px; color:#666; text-align:right;">
        Tokens: {priority_result.metrics.accumulated_usage.get('totalTokens', 0):,}
    </div>
</div>
"""
display(HTML(priority_html))

⏱️  [2026-06-06 15:46:31] Cell 3: Agent 2 - Prioritization
------------------------------------------------------------
📊 [Agent 2] Prioritization Agent 실행 중...



Rank,Hypothesis,Score,Priority
🥇,"Database Connection Failure 테스트 데이터베이스 실패는 가장 큰 영향을 미칠 수 있지만, 롤백이 매우 어려운 편이며 시스템 전체에 큰 영향을 주기 때문에 최우선 테스트에서 제외되었",6.7,
🥈,"Pod CPU Stress 테스트 CPU 스트레스는 시스템 리소스를 제한하여 성능 저하를 유발하지만, 일반적으로 롤백이 쉬운 편이며 영향도가 낮아 안전성 점수가 높습니다.",6.1,
🥉,Pod Memory Stress 테스트 메모리 스트레스는 CPU보다 더 예측 가능성이 낮으며 롤백이 어려운 경우가 많습니다. 그러나 영향도와 안전성은 낮지 않아 두 번째로 우선 순위를,5.7,
4,Network Packet Loss 테스트 네트워크 패킷 손실은 실제 발생 가능성은 높지만 롤백이 매우 어렵거나 예측 가능성이 낮기 때문에 안전성 점수가 낮습니다.,5.4,
5,Resource Quota Exceeded 테스트 리소스 할당 한도 초과 실험은 영향도가 낮지만 안전성은 낮습니다. 하지만 시스템의 자동 복구 능력에 대한 인사이트를 주므로 3번째 우선순위에 배,5.2,


In [13]:
"""
=============================================================================
Cell 4: Agent 3 - Experiment Design Agent (실험 설계 에이전트)
=============================================================================
우선순위가 높은 가설을 AWS FIS 실험 템플릿으로 변환합니다.
"""

print_cell_header("Cell 4: Agent 3 - Experiment Design")
design_agent = Agent(
    model=bedrock_model,
    system_prompt="""당신은 AWS FIS 실험 설계 전문가입니다.

역할:
- 카오스 엔지니어링 가설을 AWS FIS 실험 템플릿으로 변환합니다.
- 안전 가드레일 5원칙을 반드시 준수합니다.

안전 가드레일 원칙:
1. 타깃 사전 검증: 리소스 존재 여부 확인
2. 서비스별 안전 기준: 최소 레플리카 수 확인
3. 위험 액션 자동 배제: zonal-autoshift 등 차단
4. 폭발 반경 제어: 단일 서비스만 타깃
5. 안전 우선 순위 배치: 저위험 실험부터 실행

출력 형식 (FIS 템플릿):
{
    "experiment_templates": [
        {
            "hypothesis_id": "H-XXX",
            "template_name": "chaos-eks-xxx",
            "description": "실험 설명",
            "action": {
                "name": "FIS 액션 ID",
                "parameters": {}
            },
            "targets": {
                "type": "타깃 유형",
                "selector": "선택자",
                "filters": []
            },
            "stop_conditions": ["중단 조건"],
            "duration": "PT5M",
            "safety_validation": {
                "min_replicas_check": true,
                "blast_radius": "single_pod",
                "rollback_plan": "자동 복구"
            }
        }
    ]
}
""",
    tools=[get_fis_actions, create_fis_experiment, save_to_database],
    callback_handler=None
)

priority_output = priority_result.message["content"][0]["text"] if priority_result.message else ""

print("🎯 [Agent 3] Experiment Design Agent 실행 중...\n")
design_result = design_agent(
    f"""다음 우선순위 결과에서 상위 2개 가설에 대해 FIS 실험을 생성해주세요.

1. get_fis_actions 도구로 사용 가능한 EKS FIS 액션을 확인
2. 안전 가드레일 5원칙을 검증 (위반 시 스킵)
3. create_fis_experiment 도구로 실제 FIS 실험 템플릿 생성

대상 네임스페이스: retail-store
사용 가능한 라벨: app.kubernetes.io/name=<서비스명>

우선순위 결과:
{priority_output}

결과를 save_to_database 도구로 experiment 테이블에 저장하세요.

최종 응답은 반드시 JSON만 출력하세요:
{{"experiment_templates":[{{"hypothesis_id":"H-XXX","template_name":"이름","description":"설명","action_id":"FIS액션","target_service":"서비스","target_selector":"라벨","duration":"PT5M","blast_radius":"범위","rollback_plan":"복구방법","guardrails_passed":["통과항목"],"guardrails_failed":[],"status":"CREATED/SKIPPED"}}]}}"""
)

print("\n" + "="*60)
print("🎯 실험 설계 완료")
print("="*60)

# Cell 4 visualization addition - append after design_result
# This gets appended to the end of cell 4


# --- 실험 설계 시각화 ---
d_text = design_result.message["content"][0]["text"] if design_result.message["content"] else "{}"
d_match = re.search(r'\{.*\}', d_text, re.DOTALL)
try:
    design_data = json.loads(d_match.group()) if d_match else {}
except:
    design_data = {}
templates = design_data.get("experiment_templates", [])

design_cards = ""
for idx, exp in enumerate(templates, 1):
    action_id = exp.get("action_id", exp.get("action", {}).get("name", "N/A") if isinstance(exp.get("action"), dict) else "N/A")
    target_sel = exp.get("target_selector", exp.get("targets", {}).get("selector", "N/A") if isinstance(exp.get("targets"), dict) else "N/A")
    status = exp.get("status", "CREATED")
    status_color = "#22c55e" if status == "CREATED" else "#888"

    guardrails_passed = exp.get("guardrails_passed", [])
    guardrails_failed = exp.get("guardrails_failed", [])
    guard_html = ""
    for g in guardrails_passed:
        guard_html += f'<span style="background:#22c55e22; color:#22c55e; padding:2px 6px; border-radius:3px; font-size:10px; margin:2px;">✅ {g}</span>'
    for g in guardrails_failed:
        guard_html += f'<span style="background:#ff6b6b22; color:#ff6b6b; padding:2px 6px; border-radius:3px; font-size:10px; margin:2px;">❌ {g}</span>'

    design_cards += (
        f'<div style="background:rgba(72,219,251,0.03); border:1px solid rgba(72,219,251,0.2); border-radius:10px; padding:16px; margin:10px 0;">'
        f'<div style="display:flex; justify-content:space-between; align-items:center;">'
        f'<div style="font-weight:700; font-size:14px;">🧪 실험 #{idx}: {exp.get("template_name", "N/A")}</div>'
        f'<span style="background:{status_color}22; color:{status_color}; padding:3px 8px; border-radius:4px; font-size:11px; font-weight:700;">{status}</span></div>'
        f'<div style="font-size:12px; color:#aaa; margin:8px 0;">{exp.get("description", "")}</div>'
        f'<div style="display:grid; grid-template-columns:1fr 1fr 1fr 1fr; gap:8px; margin:10px 0; font-size:11px;">'
        f'<div style="background:rgba(0,0,0,0.2); padding:8px; border-radius:4px;"><b style="color:#888;">🎯 Target</b><br><span style="color:#48dbfb; font-family:monospace;">{exp.get("target_service", "N/A")}</span></div>'
        f'<div style="background:rgba(0,0,0,0.2); padding:8px; border-radius:4px;"><b style="color:#888;">⚡ Action</b><br><span style="color:#feca57; font-family:monospace; font-size:10px;">{action_id}</span></div>'
        f'<div style="background:rgba(0,0,0,0.2); padding:8px; border-radius:4px;"><b style="color:#888;">💥 Blast</b><br><span style="color:#ff9ff3;">{exp.get("blast_radius", "N/A")}</span></div>'
        f'<div style="background:rgba(0,0,0,0.2); padding:8px; border-radius:4px;"><b style="color:#888;">🔄 Rollback</b><br><span style="color:#22c55e;">{exp.get("rollback_plan", "N/A")}</span></div></div>'
        f'<div style="margin-top:8px;"><b style="color:#888; font-size:11px;">🛡️ Guardrails:</b><div style="margin-top:4px; display:flex; flex-wrap:wrap; gap:4px;">{guard_html}</div></div>'
        f'</div>'
    )

if not templates:
    design_cards = f'<div style="background:rgba(0,0,0,0.2); padding:16px; border-radius:8px; font-size:12px; color:#ccc; white-space:pre-wrap; line-height:1.6;">{d_text[:2000]}</div>'

design_html = (
    '<div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif; background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); color:#fff; padding:24px; border-radius:14px; margin:10px 0;">'
    '<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">'
    '<h3 style="margin:0; font-size:18px;">🎯 Agent 3: Experiment Design</h3>'
    f'<span style="background:rgba(72,219,251,0.15); color:#48dbfb; padding:4px 12px; border-radius:12px; font-size:12px;">✅ {len(templates)}개 설계</span></div>'
    '<div style="background:rgba(0,0,0,0.2); border-radius:8px; padding:12px; margin-bottom:16px; font-size:11px; color:#888;">'
    '<b style="color:#feca57;">🛡️ 안전 가드레일 5원칙:</b> '
    '① 타깃 사전 검증 → ② 서비스별 안전 기준 → ③ 위험 액션 배제 → ④ 폭발 반경 제어 → ⑤ 안전 우선 배치</div>'
    f'{design_cards}'
    f'<div style="margin-top:14px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); font-size:11px; color:#666; text-align:right;">'
    f'Tokens: {design_result.metrics.accumulated_usage.get("totalTokens", 0):,}</div></div>'
)
display(HTML(design_html))


⏱️  [2026-06-06 15:46:43] Cell 4: Agent 3 - Experiment Design
------------------------------------------------------------
🎯 [Agent 3] Experiment Design Agent 실행 중...


🎯 실험 설계 완료


In [14]:
"""
=============================================================================
Cell 5: Agent 4 - Experiment Execution Agent (실험 실행 에이전트)
=============================================================================
설계된 FIS 실험을 실행하고 모니터링합니다.
⚠️ dry_run=True로 실행하여 실제 장애 주입 없이 검증만 수행합니다.
"""

print_cell_header("Cell 5: Agent 4 - Experiment Execution")
execution_agent = Agent(
    model=bedrock_model,
    system_prompt="""당신은 AWS 워크로드 사전 검증 전문가입니다.

역할:
- 설계된 실험의 타깃 리소스가 실제로 존재하는지 사전 검증합니다.
- 각 서비스의 현재 상태(Pod 수, Running 여부)를 확인합니다.
- 실험 실행 준비 완료 여부를 판단합니다.

검증 항목:
1. 타깃 리소스가 현재 존재하는지 확인
2. 타깃 서비스의 현재 레플리카 수 확인
3. 모든 Pod가 Running 상태인지 확인
4. 실험 준비 완료 여부 판단

출력 형식:
{
    "execution_results": [
        {
            "experiment_id": "EXP-XXX",
            "hypothesis_id": "H-XXX",
            "status": "DRY_RUN | COMPLETED | FAILED | SKIPPED",
            "pre_checks": {"replicas_ok": true, "target_exists": true},
            "duration_seconds": 0,
            "observations": "관찰 사항",
            "metrics": {}
        }
    ]
}
""",
    tools=[discover_eks_resources, save_to_database],
    callback_handler=None
)

design_output = design_result.message["content"][0]["text"] if design_result.message else ""

print("⚡ [Agent 4] Experiment Execution Agent 실행 중...\n")
execution_result = execution_agent(
    f"""다음 생성된 FIS 실험 템플릿을 실행해주세요.

단계:
1. discover_eks_resources(target_namespace="retail-store")로 현재 Pod 상태 확인
2. 각 실험 타깃 서비스의 Pod가 존재하고 Running인지 검증
3. 실험 실행 준비 완료 여부를 판단

중요: 반드시 네임스페이스 "retail-store"를 사용하세요.

실험 정보:
{design_output}

각 실험의 결과를 save_to_database 도구로 experiment 테이블에 업데이트하세요.

최종 응답은 반드시 JSON만 출력하세요:
{{"validation_results":[{{"hypothesis_id":"H-XXX","target_service":"서비스","status":"READY/NOT_READY","pod_count":2,"all_running":true,"observations":"검증 내용"}}],"total_pods":10,"all_healthy":true,"summary":"검증 요약"}}"""
)

print("\n" + "="*60)
print("⚡ 실험 실행 완료 (dry_run)")
print("="*60)

# Cell 5 visualization addition - append after execution_result



# --- 사전 검증 결과 시각화 ---
e_text = execution_result.message["content"][0]["text"] if execution_result.message["content"] else "{}"
try:
    e_match = re.search(r'\{.*\}', e_text, re.DOTALL)
    exec_data = json.loads(e_match.group()) if e_match else {}
except:
    exec_data = {}

validations = exec_data.get("validation_results", [])
total_pods = exec_data.get("total_pods", "?")
all_healthy = exec_data.get("all_healthy", False)
summary_text = exec_data.get("summary", "")

val_cards = ""
for v in validations:
    status = v.get("status", "UNKNOWN")
    sc = "#22c55e" if status == "READY" else "#ff6b6b"
    icon = "✅" if status == "READY" else "❌"
    val_cards += (
        f'<div style="background:rgba(255,255,255,0.03); border:1px solid {sc}33; border-left:4px solid {sc}; border-radius:8px; padding:12px 16px; margin:6px 0;">'
        f'<div style="display:flex; justify-content:space-between; align-items:center;">'
        f'<div><span style="font-weight:600; font-size:13px;">{icon} {v.get("target_service", "N/A")}</span>'
        f'<span style="color:#666; font-size:11px; margin-left:8px;">{v.get("hypothesis_id", "")}</span></div>'
        f'<span style="background:{sc}22; color:{sc}; padding:2px 8px; border-radius:4px; font-size:11px; font-weight:600;">{status}</span></div>'
        f'<div style="display:flex; gap:16px; margin-top:8px; font-size:11px;">'
        f'<span style="color:#48dbfb;">Pods: {v.get("pod_count", "?")}</span>'
        f'<span style="color:{"#22c55e" if v.get("all_running") else "#ff6b6b"};">Running: {"Yes" if v.get("all_running") else "No"}</span>'
        f'<span style="color:#aaa;">{v.get("observations", "")}</span></div></div>'
    )

if not validations:
    val_cards = f'<div style="background:rgba(0,0,0,0.2); padding:16px; border-radius:8px; font-size:12px; color:#ccc; white-space:pre-wrap; line-height:1.6;">{e_text[:2000]}</div>'

health_color = "#22c55e" if all_healthy else "#feca57"
exec_html = (
    '<div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif; background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); color:#fff; padding:24px; border-radius:14px; margin:10px 0;">'
    '<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">'
    '<h3 style="margin:0; font-size:18px;">⚡ Agent 4: Pre-flight Validation</h3>'
    f'<span style="background:{health_color}22; color:{health_color}; padding:4px 12px; border-radius:12px; font-size:12px;">{"✅ All Ready" if all_healthy else "⚠️ Issues Found"}</span></div>'
    f'<div style="display:grid; grid-template-columns:1fr 1fr; gap:12px; margin-bottom:16px;">'
    f'<div style="background:rgba(0,0,0,0.2); border-radius:8px; padding:14px; text-align:center;">'
    f'<div style="font-size:11px; color:#888;">Total Pods</div>'
    f'<div style="font-size:28px; font-weight:700; color:#48dbfb;">{total_pods}</div></div>'
    f'<div style="background:rgba(0,0,0,0.2); border-radius:8px; padding:14px; text-align:center;">'
    f'<div style="font-size:11px; color:#888;">Health Status</div>'
    f'<div style="font-size:28px; font-weight:700; color:{health_color};">{"HEALTHY" if all_healthy else "WARNING"}</div></div></div>'
    f'<div style="background:rgba(0,0,0,0.15); border-radius:8px; padding:10px 14px; margin-bottom:12px; font-size:12px; color:#aaa;">'
    f'📋 {summary_text}</div>'
    f'{val_cards}'
    f'<div style="margin-top:14px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); font-size:11px; color:#666; text-align:right;">'
    f'Tokens: {execution_result.metrics.accumulated_usage.get("totalTokens", 0):,}</div></div>'
)
display(HTML(exec_html))


⏱️  [2026-06-06 15:47:22] Cell 5: Agent 4 - Experiment Execution
------------------------------------------------------------
⚡ [Agent 4] Experiment Execution Agent 실행 중...


⚡ 실험 실행 완료 (dry_run)


In [15]:
"""
=============================================================================
Cell 6: Agent 5 - Learning & Iteration Agent (학습 및 반복 에이전트)
=============================================================================
실험 결과를 분석하여 인사이트와 개선안을 도출합니다.
"""

print_cell_header("Cell 6: Agent 5 - Learning & Iteration")
learning_agent = Agent(
    model=bedrock_model,
    system_prompt="""당신은 카오스 엔지니어링 결과 분석 및 학습 전문가입니다.

역할:
- 실험 결과를 분석하여 시스템 취약점을 식별합니다.
- 발견된 취약점에 대한 개선안을 제시합니다.
- 다음 반복 실험을 위한 추가 가설을 생성합니다.
- 복원력 성숙도 평가를 수행합니다.

출력 형식:
{
    "analysis": {
        "summary": "전체 요약",
        "vulnerabilities_found": [
            {
                "id": "V-001",
                "severity": "HIGH|MEDIUM|LOW",
                "description": "취약점 설명",
                "affected_service": "영향받는 서비스",
                "recommendation": "개선 권고사항",
                "effort": "LOW|MEDIUM|HIGH"
            }
        ],
        "resilience_score": {
            "current": 0,
            "target": 0,
            "gaps": []
        },
        "next_experiments": [
            {
                "title": "다음 실험 제목",
                "rationale": "실험 필요성"
            }
        ],
        "architectural_recommendations": ["아키텍처 개선안"]
    }
}
""",
    tools=[save_to_database],
    callback_handler=None
)

execution_output = execution_result.message["content"][0]["text"] if execution_result.message else ""

print("📈 [Agent 5] Learning & Iteration Agent 실행 중...\n")
learning_result = learning_agent(
    f"""다음 실험 결과를 종합적으로 분석해주세요.

분석 관점:
1. 발견된 취약점과 심각도 분류
2. 각 취약점에 대한 구체적 개선 권고사항
3. 현재 복원력 성숙도 점수 (1-10)
4. 다음 반복에서 수행할 추가 실험 제안
5. 아키텍처 수준의 개선 권고사항

특히 다음 관점에서 분석해주세요:
- emptyDir 사용 DB의 데이터 영속성 문제
- 단일 레플리카 서비스의 가용성 문제
- 서비스 간 종속성에 의한 연쇄 장애 가능성

실험 결과:
{execution_output}

이전 가설 및 설계 컨텍스트:
{hypothesis_output[:1000]}

결과를 save_to_database 도구로 result 테이블에 저장해주세요."""
)

print("\n" + "="*60)
print("📈 학습 및 분석 완료")
print("="*60)

# Cell 6 visualization addition - append after learning_result

# --- 학습 결과 시각화 ---
l_text = learning_result.message["content"][0]["text"] if learning_result.message["content"] else "{}"

# JSON 파싱 시도
try:
    l_match = re.search(r'\{.*\}', l_text, re.DOTALL)
    learn_data = json.loads(l_match.group()) if l_match else {}
    analysis = learn_data.get("analysis", learn_data)
except:
    analysis = {}

# 취약점 카드
vuln_cards = ""
vulnerabilities = analysis.get("vulnerabilities_found", [])
for v in vulnerabilities:
    sev = v.get("severity", "MEDIUM")
    sev_color = {"HIGH": "#ff6b6b", "MEDIUM": "#feca57", "LOW": "#48dbfb"}.get(sev, "#888")
    vuln_cards += (
        f'<div style="background:rgba(255,255,255,0.03); border-left:3px solid {sev_color}; border-radius:6px; padding:12px; margin:6px 0;">'
        f'<div style="display:flex; justify-content:space-between; align-items:center;">'
        f'<span style="font-weight:600; font-size:12px;">{v.get("description", "N/A")}</span>'
        f'<span style="background:{sev_color}22; color:{sev_color}; padding:2px 8px; border-radius:4px; font-size:10px; font-weight:700;">{sev}</span></div>'
        f'<div style="font-size:11px; color:#aaa; margin-top:6px;">🎯 {v.get("affected_service", "N/A")} | 💡 {v.get("recommendation", "N/A")}</div>'
        f'</div>'
    )

# 복원력 점수
resilience = analysis.get("resilience_score", {})
current_score = resilience.get("current", 5)
target_score = resilience.get("target", 8)

# 다음 실험 제안
next_exp_html = ""
next_experiments = analysis.get("next_experiments", [])
for ne in next_experiments[:3]:
    next_exp_html += (
        f'<div style="background:rgba(84,160,255,0.05); border:1px solid rgba(84,160,255,0.2); border-radius:6px; padding:10px; margin:4px 0;">'
        f'<div style="font-size:12px; font-weight:600; color:#54a0ff;">{ne.get("title", "N/A")}</div>'
        f'<div style="font-size:10px; color:#888; margin-top:3px;">{ne.get("rationale", "")}</div></div>'
    )

# 아키텍처 권고
arch_html = ""
for rec in analysis.get("architectural_recommendations", [])[:4]:
    arch_html += f'<div style="font-size:11px; color:#ccc; padding:4px 0;">• {rec}</div>'

# 텍스트 폴백 (JSON 파싱 실패 시)
if not vulnerabilities and not next_experiments:
    content_block = f'<div style="background:rgba(0,0,0,0.2); padding:16px; border-radius:8px; font-size:12px; color:#ccc; white-space:pre-wrap; line-height:1.6;">{l_text[:2000]}</div>'
else:
    content_block = (
        f'<div style="display:grid; grid-template-columns:1fr 1fr; gap:16px; margin-bottom:16px;">'
        f'<div style="background:rgba(0,0,0,0.2); border-radius:8px; padding:16px; text-align:center;">'
        f'<div style="font-size:11px; color:#888;">현재 복원력 점수</div>'
        f'<div style="font-size:36px; font-weight:700; background:linear-gradient(90deg,#ff6b6b,#feca57); -webkit-background-clip:text; -webkit-text-fill-color:transparent;">{current_score}/10</div>'
        f'<div style="background:rgba(255,255,255,0.1); border-radius:4px; height:8px; margin-top:8px;"><div style="width:{current_score*10}%; height:100%; background:linear-gradient(90deg,#ff6b6b,#feca57); border-radius:4px;"></div></div>'
        f'<div style="font-size:10px; color:#888; margin-top:4px;">목표: {target_score}/10</div></div>'
        f'<div style="background:rgba(0,0,0,0.2); border-radius:8px; padding:16px; text-align:center;">'
        f'<div style="font-size:11px; color:#888;">발견된 취약점</div>'
        f'<div style="font-size:36px; font-weight:700; color:#ff6b6b;">{len(vulnerabilities)}</div>'
        f'<div style="font-size:10px; color:#888; margin-top:4px;">HIGH: {sum(1 for v in vulnerabilities if v.get("severity")=="HIGH")} | '
        f'MEDIUM: {sum(1 for v in vulnerabilities if v.get("severity")=="MEDIUM")} | '
        f'LOW: {sum(1 for v in vulnerabilities if v.get("severity")=="LOW")}</div></div></div>'
        f'<div style="margin-bottom:16px;"><b style="font-size:12px; color:#ff6b6b;">🔴 발견된 취약점</b>{vuln_cards}</div>'
        f'<div style="margin-bottom:16px;"><b style="font-size:12px; color:#54a0ff;">🔄 다음 반복 실험 제안</b><div style="margin-top:6px;">{next_exp_html}</div></div>'
        f'<div><b style="font-size:12px; color:#22c55e;">🏗️ 아키텍처 개선 권고</b><div style="margin-top:6px;">{arch_html}</div></div>'
    )

learn_html = (
    '<div style="font-family:-apple-system,BlinkMacSystemFont,sans-serif; background:linear-gradient(135deg,#1a1a2e 0%,#16213e 100%); color:#fff; padding:24px; border-radius:14px; margin:10px 0;">'
    '<div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:18px;">'
    '<h3 style="margin:0; font-size:18px;">📈 Agent 5: Learning & Iteration</h3>'
    '<span style="background:rgba(84,160,255,0.15); color:#54a0ff; padding:4px 12px; border-radius:12px; font-size:12px;">Analysis Complete</span></div>'
    f'{content_block}'
    f'<div style="margin-top:14px; padding-top:12px; border-top:1px solid rgba(255,255,255,0.08); font-size:11px; color:#666; text-align:right;">'
    f'Tokens: {learning_result.metrics.accumulated_usage.get("totalTokens", 0):,}</div>'
    '</div>'
)
display(HTML(learn_html))


⏱️  [2026-06-06 15:47:34] Cell 6: Agent 5 - Learning & Iteration
------------------------------------------------------------
📈 [Agent 5] Learning & Iteration Agent 실행 중...


📈 학습 및 분석 완료


In [16]:
"""
=============================================================================
Cell 7: 파이프라인 실행 요약 (시각화)
=============================================================================
"""
print_cell_header("Cell 7: Pipeline Summary")
from IPython.display import display, HTML

agents_info = [
    ("🔬 Hypothesis Generator", hypothesis_result, "#ff6b6b"),
    ("📊 Prioritization Agent", priority_result, "#feca57"),
    ("🎯 Experiment Design", design_result, "#48dbfb"),
    ("⚡ Experiment Execution", execution_result, "#ff9ff3"),
    ("📈 Learning & Iteration", learning_result, "#54a0ff"),
]

total_input = 0
total_output = 0
rows_html = ""
for name, result, color in agents_info:
    usage = result.metrics.accumulated_usage if result.metrics else {}
    inp = usage.get("inputTokens", 0)
    out = usage.get("outputTokens", 0)
    total = inp + out
    total_input += inp
    total_output += out
    bar_width = min(total / 100, 100)
    rows_html += f"""
    <tr>
        <td style="font-weight:600;">{name}</td>
        <td style="text-align:right;">{inp:,}</td>
        <td style="text-align:right;">{out:,}</td>
        <td style="text-align:right; font-weight:700; color:{color};">{total:,}</td>
        <td style="width:30%;">
            <div style="background:rgba(255,255,255,0.1); border-radius:4px; height:12px; overflow:hidden;">
                <div style="width:{bar_width}%; height:100%; background:{color}; border-radius:4px;"></div>
            </div>
        </td>
    </tr>"""

summary_html = f"""
<style>
.summary-box {{
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, #0d1117 0%, #161b22 100%);
    color: #fff;
    padding: 25px;
    border-radius: 12px;
    border: 1px solid rgba(255,255,255,0.1);
}}
.summary-box h2 {{
    margin: 0 0 20px;
    font-size: 20px;
}}
.summary-box table {{
    width: 100%;
    border-collapse: collapse;
}}
.summary-box th {{
    text-align: left;
    padding: 8px;
    font-size: 11px;
    color: #8b949e;
    border-bottom: 1px solid rgba(255,255,255,0.1);
}}
.summary-box td {{
    padding: 10px 8px;
    font-size: 13px;
    border-bottom: 1px solid rgba(255,255,255,0.05);
}}
.total-row {{ background: rgba(255,255,255,0.03); font-weight: 700; }}
</style>
<div class="summary-box">
    <h2>⚡ Pipeline Execution Summary</h2>
    <table>
        <tr><th>Agent</th><th style="text-align:right">Input Tokens</th><th style="text-align:right">Output Tokens</th><th style="text-align:right">Total</th><th>Usage</th></tr>
        {rows_html}
        <tr class="total-row">
            <td>합계</td>
            <td style="text-align:right">{total_input:,}</td>
            <td style="text-align:right">{total_output:,}</td>
            <td style="text-align:right; color:#feca57;">{total_input + total_output:,}</td>
            <td></td>
        </tr>
    </table>
</div>
"""
display(HTML(summary_html))

⏱️  [2026-06-06 15:48:01] Cell 7: Pipeline Summary
------------------------------------------------------------


Agent,Input Tokens,Output Tokens,Total,Usage
🔬 Hypothesis Generator,"6,638","2,525","9,163",
📊 Prioritization Agent,"2,015",724,"2,739",
🎯 Experiment Design,"10,416",878,"11,294",
⚡ Experiment Execution,"5,998",469,"6,467",
📈 Learning & Iteration,"3,114",954,"4,068",
합계,"28,181","5,550","33,731",


In [17]:
"""
=============================================================================
Cell 8: 실험 결과 시각화 대시보드
=============================================================================
전체 파이프라인 실행 결과를 시각적으로 보여줍니다.
"""
print_cell_header("Cell 8: Result Dashboard")
import json
import pathlib
from IPython.display import display, HTML, Markdown

output_dir = pathlib.Path("output")

# --- 데이터 로드 ---
def load_output(filename):
    filepath = output_dir / filename
    if not filepath.exists():
        return []
    items = []
    for block in filepath.read_text(encoding="utf-8").split("---\n"):
        block = block.strip()
        if not block:
            continue
        try:
            data = json.loads(block)
            if isinstance(data, list):
                items.extend(data)
            elif isinstance(data, dict):
                items.append(data)
        except json.JSONDecodeError:
            pass
    return items

hypotheses = load_output("hypothesis.json")
experiments = load_output("experiment.json")
results = load_output("result.json")

# 우선순위 데이터 분리
prioritized = [h for h in hypotheses if "scores" in h]
raw_hypotheses = [h for h in hypotheses if "scores" not in h]

# --- 대시보드 HTML 생성 ---
dashboard_html = """
<style>
.chaos-dashboard {
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif;
    background: linear-gradient(135deg, #1a1a2e 0%, #16213e 50%, #0f3460 100%);
    color: #fff;
    padding: 30px;
    border-radius: 16px;
    margin: 10px 0;
}
.chaos-dashboard h1 {
    text-align: center;
    font-size: 28px;
    margin-bottom: 5px;
    background: linear-gradient(90deg, #ff6b6b, #feca57, #48dbfb);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.chaos-dashboard .subtitle {
    text-align: center;
    color: #a0a0a0;
    margin-bottom: 30px;
    font-size: 14px;
}
.pipeline-flow {
    display: flex;
    justify-content: center;
    align-items: center;
    gap: 8px;
    margin: 25px 0;
    flex-wrap: wrap;
}
.pipeline-step {
    background: rgba(255,255,255,0.08);
    border: 1px solid rgba(255,255,255,0.15);
    border-radius: 12px;
    padding: 12px 16px;
    text-align: center;
    min-width: 120px;
    transition: all 0.3s;
}
.pipeline-step.active {
    border-color: #48dbfb;
    background: rgba(72, 219, 251, 0.1);
    box-shadow: 0 0 15px rgba(72, 219, 251, 0.3);
}
.pipeline-step .icon { font-size: 24px; }
.pipeline-step .label { font-size: 11px; color: #ccc; margin-top: 4px; }
.pipeline-arrow { color: #48dbfb; font-size: 20px; }
.section-title {
    font-size: 18px;
    font-weight: 600;
    margin: 30px 0 15px;
    padding-left: 12px;
    border-left: 3px solid #ff6b6b;
}
.hyp-card {
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 10px;
    padding: 16px;
    margin: 10px 0;
    transition: all 0.3s;
}
.hyp-card:hover {
    border-color: #feca57;
    background: rgba(254, 202, 87, 0.05);
}
.hyp-card .hyp-id {
    display: inline-block;
    background: #ff6b6b;
    color: #fff;
    padding: 2px 8px;
    border-radius: 4px;
    font-size: 11px;
    font-weight: 700;
}
.hyp-card .hyp-title {
    font-size: 15px;
    font-weight: 600;
    margin: 8px 0 4px;
}
.hyp-card .hyp-desc { font-size: 12px; color: #aaa; }
.hyp-card .hyp-meta {
    display: flex;
    gap: 12px;
    margin-top: 10px;
    font-size: 11px;
}
.hyp-card .meta-tag {
    background: rgba(255,255,255,0.08);
    padding: 3px 8px;
    border-radius: 4px;
    color: #48dbfb;
}
.score-table {
    width: 100%;
    border-collapse: collapse;
    margin: 15px 0;
}
.score-table th {
    background: rgba(255,107,107,0.2);
    padding: 10px;
    text-align: left;
    font-size: 12px;
    border-bottom: 1px solid rgba(255,255,255,0.1);
}
.score-table td {
    padding: 10px;
    border-bottom: 1px solid rgba(255,255,255,0.05);
    font-size: 13px;
}
.score-bar {
    height: 8px;
    border-radius: 4px;
    background: rgba(255,255,255,0.1);
    position: relative;
    overflow: hidden;
}
.score-bar-fill {
    height: 100%;
    border-radius: 4px;
    transition: width 0.5s;
}
.severity-high { color: #ff6b6b; font-weight: 700; }
.severity-medium { color: #feca57; font-weight: 700; }
.severity-low { color: #48dbfb; font-weight: 700; }
.experiment-card {
    background: rgba(72, 219, 251, 0.05);
    border: 1px solid rgba(72, 219, 251, 0.2);
    border-radius: 10px;
    padding: 16px;
    margin: 10px 0;
}
.experiment-card .exp-action {
    font-family: monospace;
    background: rgba(0,0,0,0.3);
    padding: 4px 8px;
    border-radius: 4px;
    font-size: 12px;
    color: #48dbfb;
}
.stats-grid {
    display: grid;
    grid-template-columns: repeat(auto-fit, minmax(150px, 1fr));
    gap: 15px;
    margin: 20px 0;
}
.stat-card {
    background: rgba(255,255,255,0.05);
    border: 1px solid rgba(255,255,255,0.1);
    border-radius: 10px;
    padding: 16px;
    text-align: center;
}
.stat-card .stat-number {
    font-size: 32px;
    font-weight: 700;
    background: linear-gradient(90deg, #48dbfb, #feca57);
    -webkit-background-clip: text;
    -webkit-text-fill-color: transparent;
}
.stat-card .stat-label { font-size: 11px; color: #aaa; margin-top: 4px; }
</style>

<div class="chaos-dashboard">
    <h1>Multi-Agent Chaos Engineering</h1>
    <div class="subtitle">5-Agent Pipeline Execution Report</div>
    
    <div class="pipeline-flow">
        <div class="pipeline-step active"><div class="icon">🔬</div><div class="label">Hypothesis</div></div>
        <div class="pipeline-arrow">→</div>
        <div class="pipeline-step active"><div class="icon">📊</div><div class="label">Prioritize</div></div>
        <div class="pipeline-arrow">→</div>
        <div class="pipeline-step active"><div class="icon">🎯</div><div class="label">Design</div></div>
        <div class="pipeline-arrow">→</div>
        <div class="pipeline-step active"><div class="icon">⚡</div><div class="label">Execute</div></div>
        <div class="pipeline-arrow">→</div>
        <div class="pipeline-step active"><div class="icon">📈</div><div class="label">Learn</div></div>
    </div>
    
    <div class="stats-grid">
        <div class="stat-card">
            <div class="stat-number">""" + str(len(raw_hypotheses)) + """</div>
            <div class="stat-label">가설 생성됨</div>
        </div>
        <div class="stat-card">
            <div class="stat-number">""" + str(len(prioritized)) + """</div>
            <div class="stat-label">우선순위 평가됨</div>
        </div>
        <div class="stat-card">
            <div class="stat-number">""" + str(len(experiments)) + """</div>
            <div class="stat-label">실험 설계됨</div>
        </div>
        <div class="stat-card">
            <div class="stat-number">5</div>
            <div class="stat-label">에이전트 협업</div>
        </div>
    </div>
"""

# 가설 카드들
dashboard_html += '<div class="section-title">🔬 생성된 가설 (Hypotheses)</div>'
for h in raw_hypotheses:
    domain_colors = {"computing": "#ff6b6b", "data": "#feca57", "network": "#48dbfb", "dependency": "#ff9ff3", "resource": "#54a0ff"}
    domain = h.get("failure_domain", "unknown")
    color = domain_colors.get(domain, "#aaa")
    dashboard_html += f"""
    <div class="hyp-card">
        <span class="hyp-id">{h.get('id', 'N/A')}</span>
        <div class="hyp-title">{h.get('title', 'N/A')}</div>
        <div class="hyp-desc">{h.get('hypothesis', h.get('description', ''))}</div>
        <div class="hyp-meta">
            <span class="meta-tag" style="color:{color}">📁 {domain}</span>
            <span class="meta-tag">🎯 {h.get('service', 'N/A')}</span>
            <span class="meta-tag">💥 {h.get('blast_radius', 'N/A')}</span>
            <span class="meta-tag" style="color:#54a0ff">⚡ {h.get('suggested_fis_action', 'N/A')}</span>
        </div>
    </div>"""

# 우선순위 테이블
if prioritized:
    dashboard_html += '<div class="section-title">📊 우선순위 평가 (Priority Ranking)</div>'
    sorted_p = sorted(prioritized, key=lambda x: x.get("total_score", 0), reverse=True)
    dashboard_html += '<table class="score-table"><tr><th>#</th><th>가설</th><th>Impact</th><th>Likelihood</th><th>Safety</th><th>Learning</th><th>Total</th></tr>'
    for i, p in enumerate(sorted_p, 1):
        scores = p.get("scores", {})
        total = p.get("total_score", 0)
        bar_color = "#ff6b6b" if total >= 8 else "#feca57" if total >= 7 else "#48dbfb"
        dashboard_html += f"""<tr>
            <td><strong>#{i}</strong></td>
            <td>{p.get('title', p.get('id', 'N/A'))}</td>
            <td>{scores.get('impact', '-')}/10</td>
            <td>{scores.get('likelihood', '-')}/10</td>
            <td>{scores.get('safety', '-')}/10</td>
            <td>{scores.get('learning_value', '-')}/10</td>
            <td><strong style="color:{bar_color}">{total:.1f}</strong></td>
        </tr>"""
    dashboard_html += '</table>'

# 실험 설계
if experiments:
    dashboard_html += '<div class="section-title">🎯 설계된 FIS 실험 (Experiment Templates)</div>'
    exp_list = experiments[0].get("experiment_templates", []) if isinstance(experiments[0], dict) and "experiment_templates" in experiments[0] else experiments
    for exp in exp_list:
        action_name = exp.get("action", {}).get("name", "N/A") if isinstance(exp.get("action"), dict) else exp.get("action", "N/A")
        dashboard_html += f"""
        <div class="experiment-card">
            <div style="display:flex; justify-content:space-between; align-items:center;">
                <strong>{exp.get('template_name', exp.get('hypothesis_id', 'N/A'))}</strong>
                <span class="exp-action">{action_name}</span>
            </div>
            <div style="font-size:12px; color:#aaa; margin-top:8px;">{exp.get('description', '')}</div>
            <div style="margin-top:10px; font-size:11px;">
                <span class="meta-tag">🎯 {exp.get('targets', {}).get('selector', 'N/A')}</span>
                <span class="meta-tag">⏱ {exp.get('duration', 'PT5M')}</span>
                <span class="meta-tag">💥 {exp.get('safety_validation', {}).get('blast_radius', 'N/A')}</span>
            </div>
        </div>"""

dashboard_html += """
    <div class="section-title" style="border-left-color: #48dbfb;">🏗️ Architecture Under Test</div>
    <div style="text-align:center; padding:20px; background:rgba(0,0,0,0.2); border-radius:10px; font-family:monospace; font-size:13px; line-height:2;">
        <span style="color:#48dbfb">Users</span> → <span style="color:#feca57">ALB</span> → 
        <span style="background:rgba(255,107,107,0.2); padding:4px 8px; border-radius:4px;">UI Service</span> → 
        <span style="background:rgba(254,202,87,0.2); padding:4px 8px; border-radius:4px;">Catalog (MySQL)</span> → 
        <span style="background:rgba(72,219,251,0.2); padding:4px 8px; border-radius:4px;">Cart (DynamoDB)</span> → 
        <span style="background:rgba(255,159,243,0.2); padding:4px 8px; border-radius:4px;">Checkout (Redis)</span> → 
        <span style="background:rgba(84,160,255,0.2); padding:4px 8px; border-radius:4px;">Orders (PostgreSQL)</span>
    </div>
    
    <div style="text-align:center; margin-top:25px; padding-top:20px; border-top:1px solid rgba(255,255,255,0.1); font-size:11px; color:#666;">
        Powered by <strong style="color:#48dbfb">Strands Agents SDK</strong> + <strong style="color:#feca57">Amazon Bedrock</strong> + <strong style="color:#ff6b6b">AWS FIS</strong>
        <br>EKS Cluster: {cluster_name} | Region: {eks_region}
    </div>
</div>
"""

display(HTML(dashboard_html))

cleanup_guide = """
════════════════════════════════════════════════════════════════
🧹 리소스 정리 가이드 (테스트 완료 후)
════════════════════════════════════════════════════════════════

현재 생성된 AWS 리소스 (비용 발생):
  - EKS 클러스터: {cluster_name} ({eks_region})
  - EC2 노드: EKS Auto Mode 관리 (2~3개)
  - NAT Gateway: <NAT_GATEWAY_ID>
  - Elastic IP: NAT Gateway에 연결
  - IAM Role: <FIS_ROLE_NAME>

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

정리 명령어 (필요 시 실행):

# 1. Retail Store App 삭제
kubectl delete -n retail-store -f https://github.com/aws-containers/retail-store-sample-app/releases/latest/download/kubernetes.yaml

# 2. EKS 클러스터 삭제 (콘솔에서 삭제 권장)
# aws eks delete-cluster --name <CLUSTER> --region <REGION> --profile <PROFILE>

# 3. NAT Gateway 삭제 (콘솔에서)

# 4. FIS Role 삭제
# aws iam delete-role --role-name <FIS_ROLE> --profile <PROFILE>

════════════════════════════════════════════════════════════════
"""

print(cleanup_guide)

⏱️  [2026-06-06 15:48:02] Cell 8: Result Dashboard
------------------------------------------------------------


#,가설,Impact,Likelihood,Safety,Learning,Total
#1,EmptyDir 기반 DB의 데이터 손실,9/10,7/10,9/10,8/10,8.2
#2,UI 서비스의 Redis 의존성 장애,9/10,8/10,6/10,9/10,7.8
#3,PaymentService 종속성 실패,8/10,7/10,8/10,8/10,7.7
#4,Catalog DB 연결 실패 시 장애 회복 성능,8/10,7/10,7/10,8/10,7.7
#5,Compute 자원 부족,7/10,8/10,8/10,7/10,7.5
#6,Dependency Failure in Orders Service,5/10,4/10,7/10,8/10,7.5
#7,서비스 간 네트워크 지연,6/10,8/10,9/10,7/10,7.3
#8,Data Store Latency in Catalog Service,5/10,4/10,8/10,7/10,7.1
#9,단일 레플리카 서비스의 고장,7/10,6/10,8/10,7/10,7.0
#10,Carts 서비스의 CPU 리소스 과부하,7/10,6/10,8/10,7/10,7.0



════════════════════════════════════════════════════════════════
🧹 리소스 정리 가이드 (테스트 완료 후)
════════════════════════════════════════════════════════════════

현재 생성된 AWS 리소스 (비용 발생):
  - EKS 클러스터: {cluster_name} ({eks_region})
  - EC2 노드: EKS Auto Mode 관리 (2~3개)
  - NAT Gateway: <NAT_GATEWAY_ID>
  - Elastic IP: NAT Gateway에 연결
  - IAM Role: <FIS_ROLE_NAME>

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

정리 명령어 (필요 시 실행):

# 1. Retail Store App 삭제
kubectl delete -n retail-store -f https://github.com/aws-containers/retail-store-sample-app/releases/latest/download/kubernetes.yaml

# 2. EKS 클러스터 삭제 (콘솔에서 삭제 권장)
# aws eks delete-cluster --name <CLUSTER> --region <REGION> --profile <PROFILE>

# 3. NAT Gateway 삭제 (콘솔에서)

# 4. FIS Role 삭제
# aws iam delete-role --role-name <FIS_ROLE> --profile <PROFILE>

════════════════════════════════════════════════════════════════

